# 03 — Temporal LSTM (advanced feature, lectures 10–11)

The U-Net classifies each month **independently**, so pixels flicker class-to-class
from SAR noise and seasonality, producing thousands of **phantom** month-to-month
changes. A bidirectional **LSTM** reads each pixel's 12-month sequence of class
probabilities and outputs a **temporally-consistent** class per month.

- Real change = a persistent, one-way switch; seasonal noise = oscillation.
- Training data: the trained U-Net is run over the 12 monthly tiles already in the
  Vault; for sampled pixels we build sequences X:(N,12,7) with the annual WorldCover
  label as target — so the LSTM learns to hold a consistent class.
- **Honest limitation:** trained on an *annual* label, it is deliberately conservative
  on genuine intra-year onset. Stated in the report.

In [ ]:
import torch, torch.nn as nn, numpy as np
class TemporalLSTM(nn.Module):
    def __init__(s, in_dim=7, hidden=64, n_classes=7, layers=1):
        super().__init__()
        s.lstm = nn.LSTM(in_dim, hidden, num_layers=layers, batch_first=True, bidirectional=True)
        s.head = nn.Linear(2*hidden, n_classes)
    def forward(s, x):               # x: (B, 12, 7) probabilities
        h, _ = s.lstm(x); return s.head(h)   # (B, 12, 7)

# Full sequence-building + training loop is in lstm_temporal.py:
#   python3 lstm_temporal.py   (uses unet_s1_lulc.pth + the 12-month tiles; no Earth Engine)

## Executed results

```
Total sequences: X (175500, 12, 7)   y (175500,)      LSTM params: 38k
Val per-month accuracy climbed 0.578 -> 0.827 over 30 epochs.

================ TEMPORAL LSTM RESULTS ================
Avg class switches / pixel over 12 months:
   raw U-Net (per-month argmax): 1.366
   LSTM-smoothed              : 0.162
   PHANTOM CHANGE REDUCED BY  : 88.1%
Per-month accuracy vs annual label: raw 0.766 -> LSTM 0.827
======================================================
```

The LSTM **cut phantom month-to-month change by 88%** *and* **raised per-month
accuracy** (0.766 → 0.827) — it smooths the flicker and classifies more consistently
at the same time. This is the core of separating real change from seasonal noise.

*Figure saved: `lstm_switch_reduction.png`.*